
# EfficientNet-B0 Eye ROI Baseline — Deepfake Detection

**Experiment:** EfficientNet-B0 transfer-learning baseline on 224×224 combined-eye ROI images.

This notebook is designed for Google Colab and follows the project rules provided for:
- reproducibility (`seed=42`)
- existing train/val/test split preservation
- validation-only threshold selection
- atomic `last.ckpt` / `best.ckpt`
- two-stage training (frozen backbone → full fine-tuning)
- structured metrics, predictions, figures, audit files and run summary
- no silent failure on missing/invalid metadata or images

> **Important:** The notebook does **not** re-split the dataset. It uses the `split` column already present in `metadata.csv`.


In [1]:

# ============================================================
# CELL 1 — OPTIONAL PACKAGE SETUP
# ============================================================
# Colab normally already includes torch / torchvision / sklearn / pandas.
# This installs only lightweight missing utilities without forcing a torch upgrade.

import importlib.util
import subprocess
import sys

required = {
    "yaml": "pyyaml",
    "sklearn": "scikit-learn",
    "PIL": "pillow",
}

missing = [pip_name for module_name, pip_name in required.items()
           if importlib.util.find_spec(module_name) is None]

if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("Required utility packages are already available.")


Required utility packages are already available.


In [2]:

# ============================================================
# CELL 2 — IMPORTS
# ============================================================

import os
import gc
import io
import json
import math
import time
import random
import hashlib
import platform
import subprocess
import warnings
from dataclasses import dataclass, asdict
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
import yaml
from PIL import Image, ImageFile

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
)

import matplotlib.pyplot as plt

ImageFile.LOAD_TRUNCATED_IMAGES = False
warnings.filterwarnings("once")

print("Python      :", platform.python_version())
print("PyTorch    :", torch.__version__)
try:
    import torchvision
    print("Torchvision:", torchvision.__version__)
except Exception:
    pass
print("CUDA       :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU        :", torch.cuda.get_device_name(0))


Python      : 3.12.13
PyTorch    : 2.11.0+cu128
Torchvision: 0.26.0+cu128
CUDA       : True
GPU        : Tesla T4


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [3]:

# ============================================================
# CELL 3 — CONFIGURATION
# ============================================================

@dataclass(frozen=True)
class Config:
    seed: int = 42
    image_size: int = 224

    # Training
    batch_size: int = 32
    num_workers: int = 2
    frozen_epochs: int = 5
    finetune_epochs: int = 15
    frozen_lr: float = 1e-3
    finetune_lr: float = 1e-4
    weight_decay: float = 1e-4
    patience: int = 4
    grad_clip_norm: float = 1.0

    # Binary model
    dropout: float = 0.2

    # Validation threshold search
    threshold_min: float = 0.05
    threshold_max: float = 0.95
    threshold_steps: int = 181

    # Data
    accepted_statuses: Tuple[str, ...] = ("ok", "success")
    positive_label: str = "fake"
    negative_label: str = "real"

    # Resume
    resume: bool = True

    # AMP: kept enabled on CUDA; compatibility wrapper is used below.
    amp: bool = True

CONFIG = Config()
CONFIG


Config(seed=42, image_size=224, batch_size=32, num_workers=2, frozen_epochs=5, finetune_epochs=15, frozen_lr=0.001, finetune_lr=0.0001, weight_decay=0.0001, patience=4, grad_clip_norm=1.0, dropout=0.2, threshold_min=0.05, threshold_max=0.95, threshold_steps=181, accepted_statuses=('ok', 'success'), positive_label='fake', negative_label='real', resume=True, amp=True)

In [4]:

# ============================================================
# CELL 4 — REPRODUCIBILITY
# ============================================================

def seed_everything(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Deterministic behavior is preferred for this baseline.
    # warn_only=True avoids crashing if a CUDA operation has no deterministic path.
    torch.use_deterministic_algorithms(True, warn_only=True)
    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True

seed_everything(CONFIG.seed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = bool(CONFIG.amp and DEVICE.type == "cuda")

print("DEVICE      :", DEVICE)
print("AMP_ENABLED :", AMP_ENABLED)


DEVICE      : cuda
AMP_ENABLED : True


In [5]:

# ============================================================
# CELL 5 — MOUNT GOOGLE DRIVE
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError as e:
    raise RuntimeError(
        "This notebook is intended for Google Colab. "
        "google.colab could not be imported."
    ) from e

DRIVE_ROOT = Path("/content/drive")
assert DRIVE_ROOT.exists(), "Google Drive mount failed."
print("Drive mounted:", DRIVE_ROOT)


Mounted at /content/drive
Drive mounted: /content/drive


In [10]:
# ============================================================
# CELL 6 — FIXED INPUT / OUTPUT PATHS
# ============================================================

from pathlib import Path

# ------------------------------------------------------------
# PROJECT ROOT
# ------------------------------------------------------------

DENEY1_ROOT = Path(
    "/content/drive/MyDrive/"
    "AISC DeepFake Çalışmaları/"
    "Deneyler/"
    "Kader/"
    "Deney 1"
)

# ------------------------------------------------------------
# INPUT — EYE ROI DATA
# ------------------------------------------------------------

GOZ_ROOT = DENEY1_ROOT / "Göz"

ROI_ROOT = GOZ_ROOT / "eye_roi_output"

METADATA_PATH = ROI_ROOT / "metadata.csv"

# ------------------------------------------------------------
# OUTPUT — DENEY 1 / SONUÇLAR
# ------------------------------------------------------------

RESULTS_ROOT = DENEY1_ROOT / "Sonuçlar"

# ------------------------------------------------------------
# VALIDATION
# ------------------------------------------------------------

if not DENEY1_ROOT.is_dir():
    raise FileNotFoundError(
        f"Deney 1 folder not found:\n{DENEY1_ROOT}"
    )

if not GOZ_ROOT.is_dir():
    raise FileNotFoundError(
        f"Göz folder not found:\n{GOZ_ROOT}"
    )

if not ROI_ROOT.is_dir():
    raise FileNotFoundError(
        f"eye_roi_output folder not found:\n{ROI_ROOT}"
    )

if not METADATA_PATH.is_file():
    raise FileNotFoundError(
        f"metadata.csv not found:\n{METADATA_PATH}"
    )

# Sonuçlar klasörü yoksa oluştur.
RESULTS_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# FINAL REPORT
# ------------------------------------------------------------

print("=" * 80)
print("PATH CONFIGURATION OK")
print("=" * 80)

print("\nDENEY1_ROOT:")
print(DENEY1_ROOT)

print("\nGOZ_ROOT:")
print(GOZ_ROOT)

print("\nROI_ROOT:")
print(ROI_ROOT)

print("\nMETADATA_PATH:")
print(METADATA_PATH)

print("\nRESULTS_ROOT:")
print(RESULTS_ROOT)

print("\nChecks:")
print("Deney 1 exists :", DENEY1_ROOT.exists())
print("Göz exists     :", GOZ_ROOT.exists())
print("ROI exists     :", ROI_ROOT.exists())
print("Metadata exists:", METADATA_PATH.exists())
print("Results exists :", RESULTS_ROOT.exists())

PATH CONFIGURATION OK

DENEY1_ROOT:
/content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1

GOZ_ROOT:
/content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Göz

ROI_ROOT:
/content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Göz/eye_roi_output

METADATA_PATH:
/content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Göz/eye_roi_output/metadata.csv

RESULTS_ROOT:
/content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Sonuçlar

Checks:
Deney 1 exists : True
Göz exists     : True
ROI exists     : True
Metadata exists: True
Results exists : True


In [11]:

# ============================================================
# CELL 7 — CREATE UNIQUE RUN DIRECTORY
# ============================================================

RUN_ID_BASE = datetime.now().strftime("%Y%m%d_%H%M") + "_eye_efficientnet_b0_seed42"
RUN_DIR = RESULTS_ROOT / RUN_ID_BASE

if RUN_DIR.exists():
    # Never overwrite a different experiment directory.
    suffix = 1
    while (RESULTS_ROOT / f"{RUN_ID_BASE}_r{suffix}").exists():
        suffix += 1
    RUN_DIR = RESULTS_ROOT / f"{RUN_ID_BASE}_r{suffix}"

RUN_ID = RUN_DIR.name

DIRS = {
    "configs": RUN_DIR / "configs",
    "checkpoints": RUN_DIR / "checkpoints",
    "logs": RUN_DIR / "logs",
    "metrics": RUN_DIR / "metrics",
    "predictions": RUN_DIR / "predictions",
    "figures": RUN_DIR / "figures",
    "artifacts": RUN_DIR / "artifacts",
}

for p in DIRS.values():
    p.mkdir(parents=True, exist_ok=True)

FROZEN_DIR = DIRS["checkpoints"] / "frozen"
FINETUNE_DIR = DIRS["checkpoints"] / "finetune"
FROZEN_DIR.mkdir(parents=True, exist_ok=True)
FINETUNE_DIR.mkdir(parents=True, exist_ok=True)

print("RUN_ID :", RUN_ID)
print("RUN_DIR:", RUN_DIR)


RUN_ID : 20260808_0729_eye_efficientnet_b0_seed42
RUN_DIR: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Sonuçlar/20260808_0729_eye_efficientnet_b0_seed42


In [12]:

# ============================================================
# CELL 8 — ATOMIC I/O HELPERS
# ============================================================

def atomic_write_bytes(data: bytes, target: Path) -> None:
    target.parent.mkdir(parents=True, exist_ok=True)
    tmp = target.with_suffix(target.suffix + ".tmp")
    with open(tmp, "wb") as f:
        f.write(data)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, target)

def atomic_write_text(text: str, target: Path, encoding: str = "utf-8") -> None:
    atomic_write_bytes(text.encode(encoding), target)

def atomic_json_dump(obj: Any, target: Path) -> None:
    atomic_write_text(json.dumps(obj, indent=2, ensure_ascii=False, default=str), target)

def atomic_yaml_dump(obj: Any, target: Path) -> None:
    atomic_write_text(yaml.safe_dump(obj, sort_keys=False, allow_unicode=True), target)

def atomic_csv_dump(df: pd.DataFrame, target: Path) -> None:
    payload = df.to_csv(index=False).encode("utf-8")
    atomic_write_bytes(payload, target)

def atomic_torch_save(state: dict, target: Path) -> None:
    target.parent.mkdir(parents=True, exist_ok=True)
    tmp = target.with_suffix(target.suffix + ".tmp")
    torch.save(state, tmp)

    # Integrity readback before replacement.
    loaded = torch.load(tmp, map_location="cpu", weights_only=False)
    required = {"epoch", "model_state_dict", "optimizer_state_dict"}
    if not required.issubset(loaded.keys()):
        try:
            tmp.unlink()
        finally:
            raise RuntimeError(f"Checkpoint integrity validation failed: {tmp}")

    os.replace(tmp, target)

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

atomic_yaml_dump(asdict(CONFIG), RUN_DIR / "config_resolved.yaml")
atomic_json_dump(
    {
        "run_id": RUN_ID,
        "python": platform.python_version(),
        "torch": torch.__version__,
        "device": str(DEVICE),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "platform": platform.platform(),
    },
    RUN_DIR / "environment.json",
)

try:
    freeze = subprocess.check_output(
        [sys.executable, "-m", "pip", "freeze"], text=True
    )
    atomic_write_text(freeze, RUN_DIR / "requirements_lock.txt")
except Exception as e:
    # This artifact is useful but not training-critical. Record the exact failure.
    atomic_write_text(f"pip freeze failed: {repr(e)}\n", RUN_DIR / "requirements_lock.txt")

print("Run metadata written.")


Run metadata written.


In [14]:
# ============================================================
# CELL 9 — LOAD + VALIDATE EYE ROI METADATA
# ============================================================

metadata_raw = pd.read_csv(
    METADATA_PATH,
    encoding="utf-8-sig"
)

metadata_raw.columns = [
    str(c).strip()
    for c in metadata_raw.columns
]


# ------------------------------------------------------------
# REQUIRED COLUMNS
# ------------------------------------------------------------

REQUIRED_METADATA_COLUMNS = {
    "sample_id",
    "label",
    "split",
    "status",
    "combined_eye_path",
}

missing_columns = (
    REQUIRED_METADATA_COLUMNS
    .difference(metadata_raw.columns)
)

if missing_columns:
    raise ValueError(
        "metadata.csv is missing required columns: "
        f"{sorted(missing_columns)}"
    )


# ------------------------------------------------------------
# COPY + NORMALIZE TEXT COLUMNS
# ------------------------------------------------------------

metadata = metadata_raw.copy()

# IMPORTANT:
# Do NOT convert NaN to the literal string "nan".
# pandas StringDtype preserves missing values as <NA>.
text_columns = [
    "sample_id",
    "label",
    "split",
    "status",
    "combined_eye_path",
]

for col in text_columns:
    metadata[col] = (
        metadata[col]
        .astype("string")
        .str.strip()
    )

metadata["label"] = metadata["label"].str.lower()
metadata["split"] = metadata["split"].str.lower()
metadata["status"] = metadata["status"].str.lower()


# ------------------------------------------------------------
# BASIC LABEL / SPLIT VALIDATION
# ------------------------------------------------------------

allowed_labels = {
    CONFIG.negative_label,
    CONFIG.positive_label,
}

allowed_splits = {
    "train",
    "val",
    "test",
}

observed_labels = set(
    metadata["label"]
    .dropna()
    .unique()
)

observed_splits = set(
    metadata["split"]
    .dropna()
    .unique()
)

bad_labels = sorted(
    observed_labels - allowed_labels
)

bad_splits = sorted(
    observed_splits - allowed_splits
)

if bad_labels:
    raise ValueError(
        f"Unexpected labels: {bad_labels}"
    )

if bad_splits:
    raise ValueError(
        f"Unexpected splits: {bad_splits}"
    )


# ------------------------------------------------------------
# STATUS ACCOUNTING
# ------------------------------------------------------------

accepted_statuses = {
    str(x).lower()
    for x in CONFIG.accepted_statuses
}

status_counts = (
    metadata["status"]
    .fillna("<missing>")
    .value_counts(dropna=False)
)

print("=" * 80)
print("METADATA STATUS COUNTS")
print("=" * 80)
print(status_counts)


# ------------------------------------------------------------
# AUDIT INVALID / SKIPPED ROWS
# ------------------------------------------------------------

# Rows such as status=no_face are legitimate pipeline audit rows.
# They may have no sample_id and no combined_eye_path because
# no eye ROI was generated.

accepted_status_mask = (
    metadata["status"]
    .isin(accepted_statuses)
)

audit_only = metadata.loc[
    ~accepted_status_mask
].copy()

print("\nAudit-only / skipped rows:")
print(len(audit_only))


# ------------------------------------------------------------
# VALIDATE ACCEPTED ROWS
# ------------------------------------------------------------

accepted = metadata.loc[
    accepted_status_mask
].copy()

if accepted.empty:
    raise RuntimeError(
        "No accepted ROI rows exist in metadata."
    )


# Accepted rows MUST have sample_id.
missing_sample_id_mask = (
    accepted["sample_id"].isna()
    | accepted["sample_id"].eq("")
)

if missing_sample_id_mask.any():
    bad_rows = accepted.loc[
        missing_sample_id_mask,
        [
            "label",
            "split",
            "status",
            "combined_eye_path",
        ],
    ].head(20)

    raise ValueError(
        "Accepted ROI rows with missing sample_id were found.\n"
        f"{bad_rows}"
    )


# Duplicate sample_id check ONLY on accepted rows.
duplicate_sample_mask = (
    accepted["sample_id"]
    .duplicated(keep=False)
)

if duplicate_sample_mask.any():
    dupes = (
        accepted.loc[
            duplicate_sample_mask,
            "sample_id",
        ]
        .head(20)
        .tolist()
    )

    raise ValueError(
        "Duplicate sample_id values detected among "
        f"accepted ROI rows. Examples: {dupes}"
    )


# Accepted rows MUST have combined_eye_path.
missing_eye_path_mask = (
    accepted["combined_eye_path"].isna()
    | accepted["combined_eye_path"].eq("")
)

if missing_eye_path_mask.any():
    bad_rows = accepted.loc[
        missing_eye_path_mask,
        [
            "sample_id",
            "label",
            "split",
            "status",
        ],
    ].head(20)

    raise ValueError(
        "Accepted ROI rows with missing combined_eye_path "
        f"were found:\n{bad_rows}"
    )


# ------------------------------------------------------------
# RESOLVE IMAGE PATHS
# ------------------------------------------------------------

def resolve_roi_path(
    rel_or_abs: str,
) -> Path:

    p = Path(str(rel_or_abs))

    if p.is_absolute():
        return p

    return ROI_ROOT / p


accepted["resolved_eye_path"] = (
    accepted["combined_eye_path"]
    .map(resolve_roi_path)
)

accepted["path_exists"] = (
    accepted["resolved_eye_path"]
    .map(lambda p: p.is_file())
)


# ------------------------------------------------------------
# MISSING FILE CHECK
# ------------------------------------------------------------

missing_file_rows = accepted.loc[
    ~accepted["path_exists"]
].copy()

missing_success_files = int(
    len(missing_file_rows)
)

if missing_success_files > 0:

    print("\nWARNING — accepted rows with missing image files:")
    print(
        missing_file_rows[
            [
                "sample_id",
                "combined_eye_path",
                "resolved_eye_path",
            ]
        ].head(20)
    )


# Only rows with a real image are training eligible.
eligible = accepted.loc[
    accepted["path_exists"]
].copy()

eligible = eligible.reset_index(
    drop=True
)


# ------------------------------------------------------------
# DATA ACCOUNTING
# ------------------------------------------------------------

total_inputs = int(
    len(metadata)
)

success_count = int(
    len(accepted)
)

skipped_count = int(
    len(audit_only)
)

training_eligible_success_count = int(
    len(eligible)
)


if (
    total_inputs
    != success_count + skipped_count
):
    raise RuntimeError(
        "Metadata accounting mismatch: "
        f"{total_inputs} != "
        f"{success_count} + {skipped_count}"
    )


if training_eligible_success_count == 0:
    raise RuntimeError(
        "No eligible eye ROI images were found."
    )


# ------------------------------------------------------------
# SPLIT / CLASS VALIDATION
# ------------------------------------------------------------

ct = pd.crosstab(
    eligible["split"],
    eligible["label"],
)

print("\n" + "=" * 80)
print("TRAINING-ELIGIBLE SPLIT / CLASS COUNTS")
print("=" * 80)
print(ct)


for split in [
    "train",
    "val",
    "test",
]:

    split_rows = eligible.loc[
        eligible["split"] == split
    ]

    if split_rows.empty:
        raise ValueError(
            f"Required split is absent: {split}"
        )

    split_labels = set(
        split_rows["label"]
        .dropna()
        .unique()
    )

    if split_labels != allowed_labels:
        raise ValueError(
            f"Split {split!r} must contain both classes "
            f"{sorted(allowed_labels)}, "
            f"found {sorted(split_labels)}"
        )


# ------------------------------------------------------------
# CROSS-SPLIT PATH LEAKAGE
# ------------------------------------------------------------

split_paths = {
    split: set(
        eligible.loc[
            eligible["split"] == split,
            "resolved_eye_path",
        ].astype(str)
    )
    for split in [
        "train",
        "val",
        "test",
    ]
}

path_intersections = {
    "train_val": len(
        split_paths["train"]
        & split_paths["val"]
    ),

    "train_test": len(
        split_paths["train"]
        & split_paths["test"]
    ),

    "val_test": len(
        split_paths["val"]
        & split_paths["test"]
    ),
}

if any(
    path_intersections.values()
):
    raise ValueError(
        "Cross-split duplicate eye ROI paths detected: "
        f"{path_intersections}"
    )


# ------------------------------------------------------------
# SOURCE VIDEO LEAKAGE
# ------------------------------------------------------------

true_video_level_leakage_status = (
    "NOT_VERIFIABLE_FROM_CURRENT_METADATA"
)

video_intersections = None


if "source_video" in eligible.columns:

    source_video = (
        eligible["source_video"]
        .astype("string")
        .str.strip()
    )

    valid_source_video = (
        source_video.notna()
        & source_video.ne("")
    )

    if valid_source_video.all():

        groups = {
            split: set(
                eligible.loc[
                    eligible["split"] == split,
                    "source_video",
                ].astype(str)
            )
            for split in [
                "train",
                "val",
                "test",
            ]
        }

        video_intersections = {
            "train_val": len(
                groups["train"]
                & groups["val"]
            ),

            "train_test": len(
                groups["train"]
                & groups["test"]
            ),

            "val_test": len(
                groups["val"]
                & groups["test"]
            ),
        }

        if any(
            video_intersections.values()
        ):
            raise ValueError(
                "Source-video leakage detected: "
                f"{video_intersections}"
            )

        true_video_level_leakage_status = (
            "VERIFIED_FROM_SOURCE_VIDEO_COLUMN"
        )


# ------------------------------------------------------------
# DATA ACCOUNTING REPORT
# ------------------------------------------------------------

data_accounting = {

    "run_id": RUN_ID,

    "metadata_path": str(
        METADATA_PATH
    ),

    "total_metadata_rows": (
        total_inputs
    ),

    "status_accepted_rows": (
        success_count
    ),

    "audit_only_or_skipped_rows": (
        skipped_count
    ),

    "missing_sample_id_in_all_metadata": int(
        metadata["sample_id"]
        .isna()
        .sum()
    ),

    "missing_files_among_accepted_status": (
        missing_success_files
    ),

    "training_eligible_success_count": (
        training_eligible_success_count
    ),

    "status_counts": {
        str(k): int(v)
        for k, v
        in status_counts.items()
    },

    "split_class_counts": {

        split: {

            label: int(
                (
                    (
                        eligible["split"]
                        == split
                    )
                    & (
                        eligible["label"]
                        == label
                    )
                ).sum()
            )

            for label
            in sorted(
                allowed_labels
            )
        }

        for split
        in [
            "train",
            "val",
            "test",
        ]
    },

    "cross_split_duplicate_output_path_count": int(
        sum(
            path_intersections.values()
        )
    ),

    "cross_split_path_intersections": (
        path_intersections
    ),

    "true_video_level_leakage_status": (
        true_video_level_leakage_status
    ),

    "source_video_intersections": (
        video_intersections
    ),

    "true_video_level_note": (
        "The current metadata video_id field is not "
        "treated as authoritative original source-video "
        "identity unless an explicit source_video column "
        "is available."
    ),
}


# ------------------------------------------------------------
# SAVE AUDIT FILES
# ------------------------------------------------------------

atomic_json_dump(
    data_accounting,
    RUN_DIR / "data_accounting.json",
)

atomic_csv_dump(
    eligible,
    DIRS["artifacts"]
    / "eligible_metadata.csv",
)

atomic_csv_dump(
    audit_only,
    DIRS["artifacts"]
    / "skipped_metadata.csv",
)


# ------------------------------------------------------------
# FINAL OUTPUT
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("METADATA VALIDATION PASSED")
print("=" * 80)

print(
    json.dumps(
        data_accounting,
        indent=2,
        ensure_ascii=False,
        default=str,
    )
)

METADATA STATUS COUNTS
status
ok         2986
no_face     111
Name: count, dtype: Int64

Audit-only / skipped rows:
111


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



TRAINING-ELIGIBLE SPLIT / CLASS COUNTS
label  fake  real
split            
test    156   146
train  1191  1197
val     141   155

METADATA VALIDATION PASSED
{
  "run_id": "20260808_0729_eye_efficientnet_b0_seed42",
  "metadata_path": "/content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Göz/eye_roi_output/metadata.csv",
  "total_metadata_rows": 3097,
  "status_accepted_rows": 2986,
  "audit_only_or_skipped_rows": 111,
  "missing_sample_id_in_all_metadata": 111,
  "missing_files_among_accepted_status": 0,
  "training_eligible_success_count": 2986,
  "status_counts": {
    "ok": 2986,
    "no_face": 111
  },
  "split_class_counts": {
    "train": {
      "fake": 1191,
      "real": 1197
    },
    "val": {
      "fake": 141,
      "real": 155
    },
    "test": {
      "fake": 156,
      "real": 146
    }
  },
  "cross_split_duplicate_output_path_count": 0,
  "cross_split_path_intersections": {
    "train_val": 0,
    "train_test": 0,
    "val_test": 0
  },
  "true_vi

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [15]:

# ============================================================
# CELL 10 — TRANSFORMS + DATASET
# ============================================================

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Baseline augmentation is intentionally modest.
train_transform = transforms.Compose([
    transforms.Resize(
        (CONFIG.image_size, CONFIG.image_size),
        interpolation=transforms.InterpolationMode.BICUBIC,
        antialias=True,
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.10, contrast=0.10),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize(
        (CONFIG.image_size, CONFIG.image_size),
        interpolation=transforms.InterpolationMode.BICUBIC,
        antialias=True,
    ),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

LABEL_TO_INT = {
    CONFIG.negative_label: 0,
    CONFIG.positive_label: 1,
}
INT_TO_LABEL = {v: k for k, v in LABEL_TO_INT.items()}

class EyeROIDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, transform):
        self.df = frame.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = Path(row["resolved_eye_path"])

        try:
            with Image.open(path) as img:
                img = img.convert("RGB")
                x = self.transform(img)
        except Exception as e:
            raise RuntimeError(
                f"Failed to read/transform image at dataset index={idx}, path={path}"
            ) from e

        y = torch.tensor(LABEL_TO_INT[row["label"]], dtype=torch.float32)

        return {
            "image": x,
            "label": y,
            "sample_id": str(row["sample_id"]),
            "path": str(path),
        }

split_frames = {
    s: eligible.loc[eligible["split"] == s].copy().reset_index(drop=True)
    for s in ["train", "val", "test"]
}

datasets = {
    "train": EyeROIDataset(split_frames["train"], train_transform),
    "val": EyeROIDataset(split_frames["val"], eval_transform),
    "test": EyeROIDataset(split_frames["test"], eval_transform),
}

generator = torch.Generator()
generator.manual_seed(CONFIG.seed)

loaders = {
    "train": DataLoader(
        datasets["train"],
        batch_size=CONFIG.batch_size,
        shuffle=True,
        num_workers=CONFIG.num_workers,
        pin_memory=(DEVICE.type == "cuda"),
        persistent_workers=(CONFIG.num_workers > 0),
        generator=generator,
        drop_last=False,
    ),
    "val": DataLoader(
        datasets["val"],
        batch_size=CONFIG.batch_size,
        shuffle=False,
        num_workers=CONFIG.num_workers,
        pin_memory=(DEVICE.type == "cuda"),
        persistent_workers=(CONFIG.num_workers > 0),
        drop_last=False,
    ),
    "test": DataLoader(
        datasets["test"],
        batch_size=CONFIG.batch_size,
        shuffle=False,
        num_workers=CONFIG.num_workers,
        pin_memory=(DEVICE.type == "cuda"),
        persistent_workers=(CONFIG.num_workers > 0),
        drop_last=False,
    ),
}

print({k: len(v.dataset) for k, v in loaders.items()})

# Shape sanity check
sample_batch = next(iter(loaders["train"]))
assert sample_batch["image"].ndim == 4
assert sample_batch["image"].shape[1:] == (3, CONFIG.image_size, CONFIG.image_size)
print("Batch shape:", tuple(sample_batch["image"].shape))


{'train': 2388, 'val': 296, 'test': 302}


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=3257) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()


Batch shape: (32, 3, 224, 224)


In [16]:

# ============================================================
# CELL 11 — MODEL
# ============================================================

def build_model(pretrained: bool = True) -> nn.Module:
    weights = EfficientNet_B0_Weights.DEFAULT if pretrained else None
    model = efficientnet_b0(weights=weights)

    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=CONFIG.dropout),
        nn.Linear(in_features, 1),
    )
    return model

model = build_model(pretrained=True).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(model.classifier)


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 101MB/s]


Total parameters    : 4,008,829
Trainable parameters: 4,008,829
Sequential(
  (0): Dropout(p=0.2, inplace=False)
  (1): Linear(in_features=1280, out_features=1, bias=True)
)


In [17]:

# ============================================================
# CELL 12 — AMP COMPATIBILITY + METRICS
# ============================================================

def make_grad_scaler(enabled: bool):
    if not enabled:
        return None
    # Compatible with modern and older Colab PyTorch builds.
    try:
        return torch.amp.GradScaler("cuda", enabled=True)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=True)

class AutocastContext:
    def __init__(self, enabled: bool):
        self.enabled = enabled
        self.ctx = None

    def __enter__(self):
        if not self.enabled:
            self.ctx = torch.autocast(device_type=DEVICE.type, enabled=False)
        else:
            try:
                self.ctx = torch.amp.autocast("cuda", enabled=True)
            except AttributeError:
                self.ctx = torch.cuda.amp.autocast(enabled=True)
        return self.ctx.__enter__()

    def __exit__(self, exc_type, exc_val, exc_tb):
        return self.ctx.__exit__(exc_type, exc_val, exc_tb)

def safe_roc_auc(y_true, probs):
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, probs))

def safe_ap(y_true, probs):
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(average_precision_score(y_true, probs))

def binary_metrics(y_true, probs, threshold: float) -> Dict[str, float]:
    y_true = np.asarray(y_true, dtype=int)
    probs = np.asarray(probs, dtype=float)
    preds = (probs >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, preds, labels=[0, 1]).ravel()

    specificity = tn / (tn + fp) if (tn + fp) > 0 else float("nan")

    return {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, preds)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, preds)),
        "precision": float(precision_score(y_true, preds, zero_division=0)),
        "recall": float(recall_score(y_true, preds, zero_division=0)),
        "f1": float(f1_score(y_true, preds, zero_division=0)),
        "specificity": float(specificity),
        "roc_auc": safe_roc_auc(y_true, probs),
        "average_precision": safe_ap(y_true, probs),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

def select_threshold_on_validation(y_true, probs) -> Tuple[float, pd.DataFrame]:
    thresholds = np.linspace(
        CONFIG.threshold_min,
        CONFIG.threshold_max,
        CONFIG.threshold_steps,
    )
    rows = []
    for t in thresholds:
        m = binary_metrics(y_true, probs, float(t))
        rows.append(m)

    table = pd.DataFrame(rows)

    # Primary criterion: max F1.
    # Tie-break 1: max balanced accuracy.
    # Tie-break 2: threshold nearest 0.5.
    table["distance_to_0_5"] = (table["threshold"] - 0.5).abs()
    best = table.sort_values(
        ["f1", "balanced_accuracy", "distance_to_0_5"],
        ascending=[False, False, True],
    ).iloc[0]

    return float(best["threshold"]), table.drop(columns=["distance_to_0_5"])


In [18]:

# ============================================================
# CELL 13 — CHECKPOINT RNG HELPERS
# ============================================================

def get_rng_state() -> dict:
    state = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state["cuda"] = torch.cuda.get_rng_state_all()
    return state

def set_rng_state(state: Optional[dict]) -> None:
    if not state:
        return
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    if torch.cuda.is_available() and "cuda" in state:
        torch.cuda.set_rng_state_all(state["cuda"])

def save_checkpoint(
    path: Path,
    *,
    epoch: int,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler,
    scaler,
    best_score: float,
    history: List[dict],
    stage_name: str,
) -> None:
    state = {
        "epoch": int(epoch),
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
        "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
        "best_metric_score": float(best_score),
        "history": history,
        "stage_name": stage_name,
        "config": asdict(CONFIG),
        "rng_state": get_rng_state(),
    }
    atomic_torch_save(state, path)

def load_checkpoint(
    path: Path,
    *,
    model: nn.Module,
    optimizer: Optional[torch.optim.Optimizer] = None,
    scheduler=None,
    scaler=None,
) -> dict:
    ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])

    if optimizer is not None:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    if scheduler is not None and ckpt.get("scheduler_state_dict") is not None:
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    if scaler is not None and ckpt.get("scaler_state_dict") is not None:
        scaler.load_state_dict(ckpt["scaler_state_dict"])

    set_rng_state(ckpt.get("rng_state"))
    return ckpt


In [24]:
# ============================================================
# CELL 14 — SAFE EPOCH RUNNER
# ============================================================

criterion = nn.BCEWithLogitsLoss()


def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    *,
    training: bool,
    optimizer: Optional[torch.optim.Optimizer] = None,
    scaler=None,
    amp_enabled: Optional[bool] = None,
) -> dict:

    if amp_enabled is None:
        amp_enabled = AMP_ENABLED

    if training:
        if optimizer is None:
            raise ValueError(
                "optimizer is required when training=True"
            )
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    n_items = 0

    all_labels = []
    all_probs = []
    all_sample_ids = []
    all_paths = []

    for batch_idx, batch in enumerate(loader):

        images = batch["image"].to(
            DEVICE,
            non_blocking=True,
        )

        labels = batch["label"].to(
            DEVICE,
            non_blocking=True,
        )

        # ----------------------------------------------------
        # INPUT NUMERICAL CHECK
        # ----------------------------------------------------

        if not torch.isfinite(images).all():
            raise FloatingPointError(
                f"NaN/Inf detected in input tensor "
                f"at batch {batch_idx}."
            )

        if not torch.isfinite(labels).all():
            raise FloatingPointError(
                f"NaN/Inf detected in labels "
                f"at batch {batch_idx}."
            )

        if training:
            optimizer.zero_grad(
                set_to_none=True
            )

        # ----------------------------------------------------
        # FORWARD
        # ----------------------------------------------------

        with torch.set_grad_enabled(training):

            if amp_enabled:

                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16,
                    enabled=True,
                ):
                    logits = (
                        model(images)
                        .squeeze(1)
                    )

                    loss = criterion(
                        logits,
                        labels,
                    )

            else:

                logits = (
                    model(images)
                    .squeeze(1)
                )

                loss = criterion(
                    logits,
                    labels,
                )

            # ------------------------------------------------
            # FORWARD NUMERICAL CHECKS
            # ------------------------------------------------

            if not torch.isfinite(logits).all():

                raise FloatingPointError(
                    "\nNON-FINITE LOGITS DETECTED\n"
                    f"batch_idx = {batch_idx}\n"
                    f"amp_enabled = {amp_enabled}\n"
                    f"image_min = {images.min().item():.6f}\n"
                    f"image_max = {images.max().item():.6f}\n"
                )

            if not torch.isfinite(loss):

                raise FloatingPointError(
                    "\nNON-FINITE LOSS DETECTED\n"
                    f"batch_idx = {batch_idx}\n"
                    f"loss = {loss.item()}\n"
                    f"amp_enabled = {amp_enabled}\n"
                )

            # ------------------------------------------------
            # BACKWARD
            # ------------------------------------------------

            if training:

                if scaler is not None:

                    scaler.scale(
                        loss
                    ).backward()

                    # Required before inspecting/clipping
                    # AMP-scaled gradients.
                    scaler.unscale_(
                        optimizer
                    )

                else:

                    loss.backward()

                # ------------------------------------------------
                # EXPLICIT GRADIENT AUDIT
                # ------------------------------------------------

                bad_gradients = []

                for name, param in model.named_parameters():

                    if param.grad is None:
                        continue

                    if not torch.isfinite(
                        param.grad
                    ).all():

                        bad_gradients.append(
                            name
                        )

                if bad_gradients:

                    raise FloatingPointError(
                        "\nNON-FINITE GRADIENTS DETECTED\n"
                        f"batch_idx = {batch_idx}\n"
                        f"amp_enabled = {amp_enabled}\n"
                        f"loss = {loss.item():.8f}\n"
                        "First affected parameters:\n"
                        + "\n".join(
                            bad_gradients[:20]
                        )
                    )

                # ------------------------------------------------
                # GRADIENT CLIPPING
                # ------------------------------------------------

                grad_norm = (
                    torch.nn.utils.clip_grad_norm_(
                        model.parameters(),
                        CONFIG.grad_clip_norm,
                        error_if_nonfinite=True,
                    )
                )

                # ------------------------------------------------
                # OPTIMIZER STEP
                # ------------------------------------------------

                if scaler is not None:

                    scaler.step(
                        optimizer
                    )

                    scaler.update()

                else:

                    optimizer.step()

        # ----------------------------------------------------
        # PROBABILITIES
        # ----------------------------------------------------

        probs = torch.sigmoid(
            logits.detach().float()
        )

        if not torch.isfinite(
            probs
        ).all():

            raise FloatingPointError(
                f"Non-finite probabilities "
                f"at batch {batch_idx}."
            )

        bs = images.size(0)

        total_loss += (
            float(
                loss.detach().float().item()
            )
            * bs
        )

        n_items += bs

        all_labels.extend(
            labels
            .detach()
            .cpu()
            .numpy()
            .astype(int)
            .tolist()
        )

        all_probs.extend(
            probs
            .cpu()
            .numpy()
            .astype(float)
            .tolist()
        )

        all_sample_ids.extend(
            list(
                batch["sample_id"]
            )
        )

        all_paths.extend(
            list(
                batch["path"]
            )
        )

    # --------------------------------------------------------
    # ACCOUNTING
    # --------------------------------------------------------

    if n_items != len(
        loader.dataset
    ):

        raise RuntimeError(
            "Epoch accounting mismatch: "
            f"processed={n_items}, "
            f"dataset={len(loader.dataset)}"
        )

    return {
        "loss":
            total_loss
            / max(n_items, 1),

        "labels":
            np.asarray(
                all_labels,
                dtype=int,
            ),

        "probabilities":
            np.asarray(
                all_probs,
                dtype=float,
            ),

        "sample_ids":
            all_sample_ids,

        "paths":
            all_paths,

        "count":
            int(n_items),
    }

In [25]:

# ============================================================
# CELL 15 — SMOKE TEST
# ============================================================

def smoke_test(model: nn.Module) -> None:
    print("Running 2-batch forward/backward smoke test...")

    # Temporary optimizer only for the smoke test.
    temp_optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=1e-6,
    )

    # Save exact model state so the smoke test does not alter experiment initialization.
    original_state = {
        k: v.detach().cpu().clone()
        for k, v in model.state_dict().items()
    }

    model.train()
    seen = 0

    for batch in loaders["train"]:
        images = batch["image"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        temp_optimizer.zero_grad(set_to_none=True)
        logits = model(images).squeeze(1)
        loss = criterion(logits, labels)

        if not torch.isfinite(loss):
            raise FloatingPointError("Smoke test produced non-finite loss.")

        loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG.grad_clip_norm)
        if not torch.isfinite(torch.as_tensor(grad_norm)):
            raise FloatingPointError("Smoke test produced non-finite gradients.")

        temp_optimizer.step()
        seen += 1
        print(f"  batch={seen}, loss={loss.item():.6f}")

        if seen >= 2:
            break

    if seen < 2:
        raise RuntimeError("Smoke test could not obtain 2 batches.")

    model.load_state_dict(original_state)
    model.to(DEVICE)
    del original_state, temp_optimizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("Smoke test PASSED.")

smoke_test(model)


Running 2-batch forward/backward smoke test...
  batch=1, loss=0.614146
  batch=2, loss=0.725690
Smoke test PASSED.


In [26]:

# ============================================================
# CELL 16 — TWO-STAGE TRAINING FUNCTION
# ============================================================

def train_stage(
    *,
    model: nn.Module,
    stage_name: str,
    epochs: int,
    lr: float,
    stage_dir: Path,
    use_amp: bool = True,
) -> dict:

    trainable = [p for p in model.parameters() if p.requires_grad]
    if not trainable:
        raise RuntimeError(f"No trainable parameters in stage {stage_name!r}")

    optimizer = torch.optim.AdamW(
        trainable,
        lr=lr,
        weight_decay=CONFIG.weight_decay,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
    )

    stage_amp_enabled = bool(
      use_amp
      and AMP_ENABLED
      )

    scaler = make_grad_scaler(
          stage_amp_enabled
      )

    print(
          f"[{stage_name}] AMP:",
          stage_amp_enabled
      )

    last_ckpt = stage_dir / "last.ckpt"
    best_ckpt = stage_dir / "best.ckpt"

    history = []
    start_epoch = 1
    best_score = -float("inf")
    wait = 0

    if CONFIG.resume and last_ckpt.exists():
        print(f"[{stage_name}] Resuming from {last_ckpt}")
        ckpt = load_checkpoint(
            last_ckpt,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
        )
        history = list(ckpt.get("history", []))
        start_epoch = int(ckpt["epoch"]) + 1
        best_score = float(ckpt.get("best_metric_score", -float("inf")))

        # Reconstruct early-stopping wait from history after most recent best.
        scores = [r.get("val_roc_auc", float("nan")) for r in history]
        finite_scores = [s if np.isfinite(s) else -float("inf") for s in scores]
        if finite_scores:
            best_idx = int(np.argmax(finite_scores))
            wait = max(0, len(history) - best_idx - 1)

    if start_epoch > epochs:
        print(f"[{stage_name}] Configured epoch count already reached.")
        if not best_ckpt.exists():
            raise FileNotFoundError(f"Missing best checkpoint: {best_ckpt}")
        return {
            "stage": stage_name,
            "history": history,
            "best_score": best_score,
            "best_checkpoint": str(best_ckpt),
            "last_checkpoint": str(last_ckpt),
        }

    for epoch in range(start_epoch, epochs + 1):
        t0 = time.time()

        train_out = run_epoch(
          model,
          loaders["train"],
          training=True,
          optimizer=optimizer,
          scaler=scaler,
          amp_enabled=stage_amp_enabled,
      )

        val_out = run_epoch(
          model,
          loaders["val"],
          training=False,
          amp_enabled=False,
      )

        train_m = binary_metrics(
            train_out["labels"],
            train_out["probabilities"],
            threshold=0.5,
        )
        val_m = binary_metrics(
            val_out["labels"],
            val_out["probabilities"],
            threshold=0.5,
        )

        monitor = val_m["roc_auc"]
        if not np.isfinite(monitor):
            monitor = val_m["f1"]

        scheduler.step(monitor)

        row = {
            "stage": stage_name,
            "epoch": int(epoch),
            "lr": float(optimizer.param_groups[0]["lr"]),
            "train_loss": float(train_out["loss"]),
            "val_loss": float(val_out["loss"]),
            **{f"train_{k}": v for k, v in train_m.items()},
            **{f"val_{k}": v for k, v in val_m.items()},
            "epoch_seconds": float(time.time() - t0),
        }
        history.append(row)

        print(
            f"[{stage_name}] epoch {epoch:02d}/{epochs} | "
            f"train_loss={row['train_loss']:.5f} | "
            f"val_loss={row['val_loss']:.5f} | "
            f"val_auc={row['val_roc_auc']:.5f} | "
            f"val_f1={row['val_f1']:.5f} | "
            f"lr={row['lr']:.2e}"
        )

        improved = monitor > best_score + 1e-8
        if improved:
            best_score = float(monitor)
            wait = 0
        else:
            wait += 1

        # Save last first; every completed epoch becomes resumable.
        save_checkpoint(
            last_ckpt,
            epoch=epoch,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
            best_score=best_score,
            history=history,
            stage_name=stage_name,
        )

        if improved:
            save_checkpoint(
                best_ckpt,
                epoch=epoch,
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                scaler=scaler,
                best_score=best_score,
                history=history,
                stage_name=stage_name,
            )

        atomic_csv_dump(
            pd.DataFrame(history),
            DIRS["metrics"] / f"{stage_name}_training_history.csv",
        )

        if wait >= CONFIG.patience:
            print(
                f"[{stage_name}] Early stopping: "
                f"no monitor improvement for {wait} epoch(s)."
            )
            break

    if not best_ckpt.exists():
        raise RuntimeError(f"No best checkpoint was created for stage {stage_name!r}")

    return {
        "stage": stage_name,
        "history": history,
        "best_score": best_score,
        "best_checkpoint": str(best_ckpt),
        "last_checkpoint": str(last_ckpt),
    }


In [27]:

# ============================================================
# CELL 17 — STAGE 1: FROZEN EFFICIENTNET-B0 BACKBONE
# ============================================================

# Freeze convolutional feature extractor; train only binary classifier head.
for p in model.features.parameters():
    p.requires_grad = False
for p in model.classifier.parameters():
    p.requires_grad = True

print(
    "Trainable params (frozen stage):",
    f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}",
)

frozen_summary = train_stage(
    model=model,
    stage_name="frozen",
    epochs=CONFIG.frozen_epochs,
    lr=CONFIG.frozen_lr,
    stage_dir=FROZEN_DIR,
)

atomic_json_dump(
    frozen_summary,
    DIRS["metrics"] / "frozen_training_summary.json",
)

frozen_summary


Trainable params (frozen stage): 1,281
[frozen] AMP: True
[frozen] Resuming from /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Sonuçlar/20260808_0729_eye_efficientnet_b0_seed42/checkpoints/frozen/last.ckpt


TypeError: RNG state must be a torch.ByteTensor

In [23]:

# ============================================================
# CELL 18 — STAGE 2: FULL FINE-TUNING
# ============================================================

# Always start fine-tuning from the best frozen checkpoint.
frozen_best = torch.load(
    FROZEN_DIR / "best.ckpt",
    map_location=DEVICE,
    weights_only=False,
)
model.load_state_dict(frozen_best["model_state_dict"])

for p in model.parameters():
    p.requires_grad = True

print(
    "Trainable params (fine-tune stage):",
    f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}",
)

finetune_summary = train_stage(
    model=model,
    stage_name="finetune",
    epochs=CONFIG.finetune_epochs,
    lr=CONFIG.finetune_lr,
    stage_dir=FINETUNE_DIR,
)

atomic_json_dump(
    finetune_summary,
    DIRS["metrics"] / "finetune_training_summary.json",
)

finetune_summary


Trainable params (fine-tune stage): 4,008,829


FloatingPointError: Non-finite gradient norm detected.

In [ ]:

# ============================================================
# CELL 19 — LOAD BEST FINAL MODEL + VALIDATION THRESHOLD
# ============================================================

FINAL_BEST_CKPT = FINETUNE_DIR / "best.ckpt"
if not FINAL_BEST_CKPT.exists():
    raise FileNotFoundError(FINAL_BEST_CKPT)

best_final = torch.load(
    FINAL_BEST_CKPT,
    map_location=DEVICE,
    weights_only=False,
)
model.load_state_dict(best_final["model_state_dict"])
model.eval()

val_out = run_epoch(model, loaders["val"], training=False)

best_threshold, threshold_table = select_threshold_on_validation(
    val_out["labels"],
    val_out["probabilities"],
)

val_metrics_selected = binary_metrics(
    val_out["labels"],
    val_out["probabilities"],
    best_threshold,
)

atomic_csv_dump(
    threshold_table,
    DIRS["metrics"] / "validation_threshold_search.csv",
)
atomic_json_dump(
    val_metrics_selected,
    DIRS["metrics"] / "validation_selected_threshold_metrics.json",
)

print("Validation-selected threshold:", best_threshold)
print(json.dumps(val_metrics_selected, indent=2))


In [ ]:

# ============================================================
# CELL 20 — FINAL TEST EVALUATION (ONE PASS)
# ============================================================

test_out = run_epoch(model, loaders["test"], training=False)

test_metrics = binary_metrics(
    test_out["labels"],
    test_out["probabilities"],
    best_threshold,
)

test_predictions = pd.DataFrame({
    "sample_id": test_out["sample_ids"],
    "path": test_out["paths"],
    "label_int": test_out["labels"],
    "label": [INT_TO_LABEL[int(y)] for y in test_out["labels"]],
    "prob_fake": test_out["probabilities"],
})
test_predictions["threshold"] = best_threshold
test_predictions["pred_int"] = (
    test_predictions["prob_fake"].to_numpy() >= best_threshold
).astype(int)
test_predictions["prediction"] = test_predictions["pred_int"].map(INT_TO_LABEL)
test_predictions["correct"] = (
    test_predictions["label_int"] == test_predictions["pred_int"]
)

atomic_csv_dump(
    pd.DataFrame([{"model": "EfficientNet-B0", **test_metrics}]),
    DIRS["metrics"] / "final_test_metrics.csv",
)
atomic_csv_dump(
    test_predictions,
    DIRS["predictions"] / "test_predictions.csv",
)

print(json.dumps(test_metrics, indent=2))


In [ ]:

# ============================================================
# CELL 21 — FIGURE HELPERS
# ============================================================

def save_figure(fig, filename: str) -> Path:
    path = DIRS["figures"] / filename
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)

    # Quality gate: short edge >= 600 px.
    with Image.open(path) as img:
        if min(img.size) < 600:
            raise RuntimeError(f"Figure resolution below 600px: {path} -> {img.size}")
    return path

# Collect histories
frozen_hist = pd.read_csv(DIRS["metrics"] / "frozen_training_history.csv")
finetune_hist = pd.read_csv(DIRS["metrics"] / "finetune_training_history.csv")

history_all = pd.concat([frozen_hist, finetune_hist], ignore_index=True)
history_all["global_epoch"] = np.arange(1, len(history_all) + 1)

print(history_all[[
    "stage", "epoch", "train_loss", "val_loss", "val_roc_auc", "val_f1"
]].tail())


In [ ]:

# ============================================================
# CELL 22 — TRAINING CURVES
# ============================================================

fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
ax.plot(history_all["global_epoch"], history_all["train_loss"], label="Training Loss")
ax.plot(history_all["global_epoch"], history_all["val_loss"], label="Validation Loss")
ax.set_title("EfficientNet-B0 Training and Validation Loss")
ax.set_xlabel("Global Epoch")
ax.set_ylabel("Loss")
ax.legend()
ax.grid(True, alpha=0.25)
fig.tight_layout()
save_figure(fig, "training_validation_loss_curve.png")

fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
ax.plot(history_all["global_epoch"], history_all["val_roc_auc"], label="Validation ROC-AUC")
ax.plot(history_all["global_epoch"], history_all["val_f1"], label="Validation F1")
ax.set_title("EfficientNet-B0 Validation Metrics")
ax.set_xlabel("Global Epoch")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
ax.legend()
ax.grid(True, alpha=0.25)
fig.tight_layout()
save_figure(fig, "validation_metrics_curve.png")

print("Training curves saved.")


In [ ]:

# ============================================================
# CELL 23 — ROC + PRECISION-RECALL CURVES
# ============================================================

y_test = test_out["labels"]
p_test = test_out["probabilities"]

fpr, tpr, _ = roc_curve(y_test, p_test)
roc_auc = roc_auc_score(y_test, p_test)

fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
ax.plot(fpr, tpr, label=f"EfficientNet-B0 (AUC={roc_auc:.3f})")
ax.plot([0, 1], [0, 1], linestyle="--", label="Chance")
ax.set_title("Test ROC Curve")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend()
ax.grid(True, alpha=0.25)
fig.tight_layout()
save_figure(fig, "test_roc_curve.png")

precision_curve, recall_curve, _ = precision_recall_curve(y_test, p_test)
ap = average_precision_score(y_test, p_test)

fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
ax.plot(recall_curve, precision_curve, label=f"EfficientNet-B0 (AP={ap:.3f})")
ax.set_title("Test Precision-Recall Curve")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.legend()
ax.grid(True, alpha=0.25)
fig.tight_layout()
save_figure(fig, "test_precision_recall_curve.png")

print("ROC and PR curves saved.")


In [ ]:

# ============================================================
# CELL 24 — CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_test,
    (p_test >= best_threshold).astype(int),
    labels=[0, 1],
)

fig, ax = plt.subplots(figsize=(8, 8), dpi=150)
im = ax.imshow(cm)
fig.colorbar(im, ax=ax)

ax.set_title(f"Test Confusion Matrix (Threshold={best_threshold:.3f})")
ax.set_xlabel("Predicted Label")
ax.set_ylabel("True Label")
ax.set_xticks([0, 1], labels=["Real", "Fake"])
ax.set_yticks([0, 1], labels=["Real", "Fake"])

for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=14)

fig.tight_layout()
save_figure(fig, "test_confusion_matrix.png")

print(cm)


In [ ]:

# ============================================================
# CELL 25 — PROBABILITY DISTRIBUTION + THRESHOLD ANALYSIS
# ============================================================

fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
ax.hist(
    p_test[y_test == 0],
    bins=25,
    alpha=0.6,
    label="Real",
)
ax.hist(
    p_test[y_test == 1],
    bins=25,
    alpha=0.6,
    label="Fake",
)
ax.axvline(best_threshold, linestyle="--", label=f"Threshold={best_threshold:.3f}")
ax.set_title("Test Predicted Probability Distribution")
ax.set_xlabel("Predicted Probability of Fake")
ax.set_ylabel("Count")
ax.legend()
ax.grid(True, alpha=0.25)
fig.tight_layout()
save_figure(fig, "test_probability_distribution.png")

fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
ax.plot(threshold_table["threshold"], threshold_table["f1"], label="Validation F1")
ax.plot(
    threshold_table["threshold"],
    threshold_table["balanced_accuracy"],
    label="Validation Balanced Accuracy",
)
ax.plot(
    threshold_table["threshold"],
    threshold_table["specificity"],
    label="Validation Specificity",
)
ax.axvline(best_threshold, linestyle="--", label=f"Selected={best_threshold:.3f}")
ax.set_title("Validation Threshold Analysis")
ax.set_xlabel("Threshold")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
ax.legend()
ax.grid(True, alpha=0.25)
fig.tight_layout()
save_figure(fig, "validation_threshold_analysis.png")

print("Probability and threshold figures saved.")


In [ ]:

# ============================================================
# CELL 26 — SAMPLE PREDICTIONS FIGURE
# ============================================================

def plot_prediction_samples(pred_df: pd.DataFrame, n: int = 12):
    # Prefer a mix of correct and incorrect samples.
    wrong = pred_df.loc[~pred_df["correct"]]
    correct = pred_df.loc[pred_df["correct"]]

    n_wrong = min(len(wrong), n // 2)
    n_correct = n - n_wrong

    selected = pd.concat([
        wrong.head(n_wrong),
        correct.head(n_correct),
    ], ignore_index=True)

    if len(selected) == 0:
        return None

    cols = 4
    rows = math.ceil(len(selected) / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(12, 3.2 * rows), dpi=150)
    axes = np.array(axes).reshape(-1)

    for ax in axes:
        ax.axis("off")

    for ax, (_, row) in zip(axes, selected.iterrows()):
        with Image.open(row["path"]) as img:
            ax.imshow(img.convert("RGB"))
        ax.set_title(
            f"True: {row['label'].title()} | Pred: {row['prediction'].title()}\n"
            f"P(fake)={row['prob_fake']:.3f}",
            fontsize=9,
        )
        ax.axis("off")

    fig.suptitle("EfficientNet-B0 Test Sample Predictions", fontsize=14)
    fig.tight_layout()
    return save_figure(fig, "test_sample_predictions.png")

sample_fig = plot_prediction_samples(test_predictions, n=12)
print("Sample figure:", sample_fig)


In [ ]:

# ============================================================
# CELL 27 — INFERENCE RELOAD TEST
# ============================================================

def inference_reload_test() -> dict:
    reloaded = build_model(pretrained=False).to(DEVICE)
    ckpt = torch.load(FINAL_BEST_CKPT, map_location=DEVICE, weights_only=False)
    reloaded.load_state_dict(ckpt["model_state_dict"])
    reloaded.eval()

    batch = next(iter(loaders["test"]))
    x = batch["image"][: min(4, len(batch["image"]))].to(DEVICE)

    with torch.no_grad():
        logits = reloaded(x).squeeze(1)
        probs = torch.sigmoid(logits)

    if probs.ndim != 1:
        raise RuntimeError(f"Unexpected inference output shape: {tuple(probs.shape)}")
    if not torch.isfinite(probs).all():
        raise FloatingPointError("Reloaded model produced NaN/Inf.")
    if ((probs < 0) | (probs > 1)).any():
        raise ValueError("Reloaded model produced probabilities outside [0, 1].")

    result = {
        "status": "PASSED",
        "checkpoint": str(FINAL_BEST_CKPT),
        "n_samples": int(len(probs)),
        "probabilities": probs.detach().cpu().numpy().astype(float).tolist(),
    }
    return result

inference_test = inference_reload_test()
atomic_json_dump(
    inference_test,
    DIRS["metrics"] / "inference_reload_test.json",
)
print(json.dumps(inference_test, indent=2))


In [ ]:

# ============================================================
# CELL 28 — OUTPUT MANIFEST + RUN SUMMARY
# ============================================================

def build_output_manifest(root: Path) -> pd.DataFrame:
    rows = []
    for p in sorted(root.rglob("*")):
        if not p.is_file():
            continue
        # Avoid hashing very large checkpoints unnecessarily in the manifest.
        size = p.stat().st_size
        digest = sha256_file(p) if size <= 50 * 1024 * 1024 else ""
        rows.append({
            "relative_path": str(p.relative_to(root)),
            "size_bytes": int(size),
            "sha256": digest,
        })
    return pd.DataFrame(rows)

manifest = build_output_manifest(RUN_DIR)
atomic_csv_dump(manifest, RUN_DIR / "output_manifest.csv")

run_summary = {
    "run_id": RUN_ID,
    "experiment": "EfficientNet-B0 Eye ROI Transfer Learning Baseline",
    "model": "torchvision EfficientNet-B0",
    "pretrained_weights": "EfficientNet_B0_Weights.DEFAULT",
    "input": "combined eye ROI",
    "image_size": CONFIG.image_size,
    "seed": CONFIG.seed,
    "device": str(DEVICE),
    "metadata_path": str(METADATA_PATH),
    "roi_root": str(ROI_ROOT),
    "results_root": str(RESULTS_ROOT),
    "run_dir": str(RUN_DIR),
    "split_counts": data_accounting["split_class_counts"],
    "leakage_status": data_accounting["true_video_level_leakage_status"],
    "frozen_best_score": frozen_summary["best_score"],
    "finetune_best_score": finetune_summary["best_score"],
    "validation_selected_threshold": best_threshold,
    "validation_metrics": val_metrics_selected,
    "final_test_metrics": test_metrics,
    "inference_reload_test": inference_test["status"],
    "output_file_count": int(len(manifest)),
}

atomic_json_dump(run_summary, RUN_DIR / "run_summary.json")

print(json.dumps(run_summary, indent=2, ensure_ascii=False, default=str))
print("\nFINAL OUTPUT DIRECTORY:")
print(RUN_DIR)



## Expected final output

The notebook creates a unique run folder under the existing **Experiment 1 → Results** directory, for example:

```text
20260808_HHMM_eye_efficientnet_b0_seed42/
├── config_resolved.yaml
├── environment.json
├── requirements_lock.txt
├── data_accounting.json
├── run_summary.json
├── output_manifest.csv
├── checkpoints/
│   ├── frozen/
│   │   ├── last.ckpt
│   │   └── best.ckpt
│   └── finetune/
│       ├── last.ckpt
│       └── best.ckpt
├── metrics/
├── predictions/
├── figures/
├── artifacts/
└── logs/
```

The final test threshold is selected **only on the validation set**. The test set is not used for model or threshold selection.
